# Phase 1: Unified Data Preparation Pipeline

This notebook consolidates both previous implementations into one clean workflow:
- Robust preprocessing and asset-specific label tuning
- FinBERT embedding extraction
- Structured formatted records (for downstream JSONL use)
- Chronological train/validation/test splits without leakage

The pipeline outputs both:
1. `train_df`, `val_df`, `test_df` (model-friendly tabular view)
2. `train_data`, `val_data`, `test_data` (formatted nested records)

In [ ]:
import os
import json
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel
from huggingface_hub import login

# Optional Hugging Face login (recommended for stable dataset/model download).
# HF_TOKEN = os.getenv("HF_TOKEN")
HF_TOKEN = "Your HF_TOKEN"  # @ai23btech11007 & @ai23btech11033, get your own token from https://huggingface.co/settings/tokens and set it here. 
# i won't hard code it here, hehehe
if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    print("HF_TOKEN not set. Continuing without explicit login.")


In [ ]:
# Load selected assets from TheFinAI daily_news dataset.
splits = {
    "BTC": "data/BTC-00000-of-00001.parquet",
    "TSLA": "data/TSLA-00000-of-00001.parquet",
    "MSFT": "data/MSFT-00000-of-00001.parquet",
    "ETH": "data/ETH-00000-of-00001.parquet",
    "BMRN": "data/BMRN-00000-of-00001.parquet",
    "MRNA": "data/MRNA-00000-of-00001.parquet",
}

frames = [
    pd.read_parquet("hf://datasets/TheFinAI/daily_news/" + path)
    for path in splits.values()
]
df = pd.concat(frames, ignore_index=True)

print(f"Loaded rows: {len(df):,}")
df.head(2)


In [ ]:
# Asset-wise threshold tuning for balanced {-1, 0, 1} labels.
df["returns"] = df["future_price_diff"] / df["prices"]

def pick_best_tou(asset_returns, quantile_grid=None):
    if quantile_grid is None:
        quantile_grid = [i / 100 for i in range(5, 46)]  # 5%..45%

    x = asset_returns.dropna()
    if x.empty:
        return None, None

    best_tou = None
    best_score = float("inf")
    best_props = None

    for q in quantile_grid:
        tou_candidate = x.abs().quantile(q)
        labels = x.apply(
            lambda r: 1 if r > tou_candidate else (-1 if r < -tou_candidate else 0)
        )
        props = labels.value_counts(normalize=True).reindex([-1, 0, 1], fill_value=0)
        score = ((props - (1 / 3)) ** 2).sum()

        if score < best_score:
            best_score = score
            best_tou = tou_candidate
            best_props = props

    return best_tou, best_props

rows = []
for asset, g in df.groupby("asset"):
    best_tou, props = pick_best_tou(g["returns"])
    rows.append(
        {
            "asset": asset,
            "best_tou": best_tou,
            "pct_-1": props.loc[-1] if props is not None else None,
            "pct_0": props.loc[0] if props is not None else None,
            "pct_1": props.loc[1] if props is not None else None,
        }
    )

tou_by_asset = pd.DataFrame(rows).sort_values("asset").reset_index(drop=True)
tou_map = dict(zip(tou_by_asset["asset"], tou_by_asset["best_tou"]))
df["tou_asset"] = df["asset"].map(tou_map)
df["label"] = df.apply(
    lambda row: 1 if row["returns"] > row["tou_asset"] else (-1 if row["returns"] < -row["tou_asset"] else 0),
    axis=1
)

label_dist = (
    df.groupby(["asset", "label"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=[-1, 0, 1], fill_value=0)
)
label_dist


In [ ]:
# Basic cleanup and ordering.
df = df[df["returns"].notna()].copy()
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(by=["asset", "date"]).reset_index(drop=True)

def aggregate_text(text_list):
    if not isinstance(text_list, (list, np.ndarray)) or len(text_list) == 0:
        return ""
    return " [SEP] ".join(str(text) for text in text_list)

df["combined_news"] = df["news"].apply(aggregate_text)
df["combined_10k"] = df["10k"].apply(aggregate_text)
df["combined_10q"] = df["10q"].apply(aggregate_text)
df["full_daily_text"] = (
    df["combined_news"] + " " + df["combined_10k"] + " " + df["combined_10q"]
).str.strip()

def encode_momentum(value):
    s = str(value).lower()
    if "bullish" in s:
        return 1
    if "bearish" in s:
        return -1
    return 0

df["momentum_encoded"] = df["momentum"].apply(encode_momentum)

print(f"Rows after cleanup: {len(df):,}")
df[["date", "asset", "returns", "label", "momentum_encoded"]].head(3)


In [ ]:
# FinBERT embedding extraction.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Loading FinBERT on {device}...")
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModel.from_pretrained("ProsusAI/finbert").to(device)
model.eval()

def get_finbert_embeddings(text_series, batch_size=16):
    all_embeddings = []

    for i in tqdm(range(0, len(text_series), batch_size), desc="Embedding Batches"):
        batch_texts = text_series.iloc[i : i + batch_size].tolist()
        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)
            batch_embeddings = outputs.pooler_output.cpu().numpy().tolist()

        all_embeddings.extend(batch_embeddings)

    return all_embeddings

df["finbert_vector"] = get_finbert_embeddings(df["full_daily_text"], batch_size=16)
print("Embedding extraction complete.")


In [ ]:
# Build formatted nested records (pros of formatted-data version).
LOOKBACK_DAYS = float("inf")
formatted_dataset = []

def to_list_or_empty(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    return value if isinstance(value, list) else [value]

def format_filing(asset, filing_value):
    filing_list = to_list_or_empty(filing_value)
    return {asset: filing_list} if len(filing_list) > 0 else None

for asset, group in df.groupby("asset"):
    dates = group["date"].dt.strftime("%Y-%m-%d").tolist()
    prices = group["prices"].tolist()
    news = group["news"].tolist()
    filings_10k = group["10k"].tolist()
    filings_10q = group["10q"].tolist()
    momentums = group["momentum"].tolist()
    momentum_encoded = group["momentum_encoded"].tolist()
    labels = group["label"].tolist()
    finbert_vectors = group["finbert_vector"].tolist()

    for i in range(len(group)):
        start_idx = max(0, i - LOOKBACK_DAYS)
        history_price_list = [
            {"date": dates[j], "price": prices[j]}
            for j in range(start_idx, i)
        ]

        curr_news = to_list_or_empty(news[i])
        curr_10k = filings_10k[i]
        curr_10q = filings_10q[i]

        record = {
            "date": dates[i],
            "symbol": [asset],
            "price": {asset: prices[i]},
            "history_price": {asset: history_price_list},
            "news": {asset: curr_news},
            "10k": format_filing(asset, curr_10k),
            "10q": format_filing(asset, curr_10q),
            "momentum": {asset: momentums[i]},
            "momentum_encoded": momentum_encoded[i],
            "finbert_vector": finbert_vectors[i],
            "label": labels[i],
        }
        formatted_dataset.append(record)

print(f"Formatted records: {len(formatted_dataset):,}")
formatted_dataset[0]


In [ ]:
# Chronological split per asset, keeping both output formats.
def chronological_split(records, train_pct=0.7, val_pct=0.15):
    n = len(records)
    train_idx = int(n * train_pct)
    val_idx = int(n * (train_pct + val_pct))
    return records[:train_idx], records[train_idx:val_idx], records[val_idx:]

asset_records = defaultdict(list)
for row in formatted_dataset:
    asset_records[row["symbol"][0]].append(row)

train_data, val_data, test_data = [], [], []
for asset, records in asset_records.items():
    tr, va, te = chronological_split(records)
    train_data.extend(tr)
    val_data.extend(va)
    test_data.extend(te)

# Tabular views for quick modeling/debugging.
train_df = pd.DataFrame(train_data)
val_df = pd.DataFrame(val_data)
test_df = pd.DataFrame(test_data)

print(
    f"Split complete -> Train: {len(train_df):,}, "
    f"Val: {len(val_df):,}, Test: {len(test_df):,}"
)
train_df.head(2)


In [ ]:
# Optional export helpers.
def save_jsonl(records, file_path):
    with open(file_path, "w", encoding="utf-8") as f:
        for row in records:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

save_jsonl(train_data, "train.jsonl")
save_jsonl(val_data, "val.jsonl")
save_jsonl(test_data, "test.jsonl")

print("Saved: train.jsonl, val.jsonl, test.jsonl")
print("Columns in train_df:", train_df.columns.tolist())


In [ ]:
# Load saved JSONL files back into Python objects / DataFrames.
def load_jsonl(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train_data_loaded = load_jsonl("train.jsonl")
val_data_loaded = load_jsonl("val.jsonl")
test_data_loaded = load_jsonl("test.jsonl")

print(
    f"Loaded -> Train: {len(train_data_loaded):,}, "
    f"Val: {len(val_data_loaded):,}, Test: {len(test_data_loaded):,}"
)

# Optional: tabular view for quick inspection.
train_df_loaded = pd.DataFrame(train_data_loaded)
val_df_loaded = pd.DataFrame(val_data_loaded)
test_df_loaded = pd.DataFrame(test_data_loaded)

train_df_loaded.head(2)
